# Notebook: Feature Engineering & Modeling
### BTT Northstar: Minh Le, Athena Tian, Rianna Lei

This notebook builds and evaluates survival-prediction approaches for wildfire event timing:
1. A **penalized Cox proportional hazards model** as a transparent baseline.
2. A **multi-horizon XGBoost setup** to capture nonlinear patterns.

The team goal is to estimate event risk over fixed time horizons (24h, 48h, 72h) and compare models using a shared evaluation framework.

---

## I. Load Packages and Data

### Data loaded in this section
- `train.csv`: labeled training data.
- `test.csv`: holdout data for later inference/submission.
- `sample_submission.csv`: output format reference.
- `metaData.csv`: data dictionary/metadata support.

In [48]:
# Import packages
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sn
import os

# General utilities
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import brier_score_loss
from sklearn.preprocessing import StandardScaler
from lifelines.utils import concordance_index

# Model 1: 
from statsmodels.duration.hazard_regression import PHReg

# Model 2:
from xgboost import XGBClassifier

# Load data
train = pd.read_csv('../data/train.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')
meta = pd.read_csv('../data/metaData.csv')
test = pd.read_csv('../data/test.csv')

## II. Feature Engineering

### Core transformations implemented:
- **Cyclical time encoding** for hour, day-of-week, and month using sine/cosine transforms.
- **Season indicators** and interactions between seasonal and temperature-related variables.
- **Risk interaction terms** (for example weather/terrain interactions) that may improve separability.
- Missing-value handling and type cleanup needed for modeling compatibility.

In [49]:
def add_time_features(df):
    """
    Add cyclical encodings for calendar/time variables.

    Why:
    - Hour 23 and hour 0 are close in reality, but far apart numerically.
    - Same logic applies to day-of-week and month.

    Parameters
    df : pd.DataFrame
        Input dataframe containing:
        - event_start_hour
        - event_start_dayofweek
        - event_start_month

    Returns: Copy of dataframe with cyclical time features added.
    """
    out = df.copy()

    # Hour of day
    out["hour_sin"] = np.sin(2 * np.pi * out["event_start_hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["event_start_hour"] / 24)

    # Day of week
    out["dow_sin"] = np.sin(2 * np.pi * out["event_start_dayofweek"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["event_start_dayofweek"] / 7)

    # Month of year
    out["month_sin"] = np.sin(2 * np.pi * out["event_start_month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["event_start_month"] / 12)

    return out


def create_features(df):
    """
    Create additional wildfire risk features for survival modeling.

    Design principles:
    Features are organized around five questions:
    1. How close is the fire to the evacuation zone?
    2. Is it moving toward the zone?
    3. How quickly is it expanding?
    4. How urgent is the threat relative to distance?
    5. How reliable is the early observation window?

    Notes
    - This function assumes the original engineered wildfire features already exist.
    - It avoids creating too many redundant nonlinear transforms.
    - It keeps feature engineering physically interpretable.

    Parameters
    df : pd.DataFrame
        Input dataframe.

    Returns: Feature-engineered dataframe with cleaned values.
    """
    out = add_time_features(df)

    # ------------------------------------------------------------------
    # 0) Base aliases
    # ------------------------------------------------------------------
    dist = out["dist_min_ci_0_5h"].clip(lower=1)                     # avoid divide-by-zero / log(0)
    speed = out["closing_speed_m_per_h"]                             # signed closing speed
    closing_pos = speed.clip(lower=0)                                # only positive approach speed
    perimeters = out["num_perimeters_0_5h"].clip(lower=0)
    area_first = out["area_first_ha"].clip(lower=0)
    radial_growth = out["radial_growth_rate_m_per_h"].clip(lower=0)

    # ------------------------------------------------------------------
    # 1) Distance / proximity features
    # ------------------------------------------------------------------

    # These capture nonlinear effects of distance without overdoing it.
    out["log_distance"] = np.log1p(dist)
    out["inv_distance"] = 1 / (dist / 1000 + 0.1)

    # Risk-zone flags
    out["zone_critical"] = (dist < 5000).astype(float)
    out["zone_warning"] = ((dist >= 5000) & (dist < 10000)).astype(float)

    # ------------------------------------------------------------------
    # 2) Fire size relative to proximity
    # ------------------------------------------------------------------

    # Convert area to a rough equivalent radius in meters:
    # area_first_ha * 10,000 converts hectares to square meters.
    fire_radius_m = np.sqrt(area_first * 10000 / np.pi)

    out["radius_to_dist"] = fire_radius_m / dist
    out["area_to_dist_ratio"] = area_first / (dist / 1000 + 0.1)
    out["log_area_dist_ratio"] = np.log1p(area_first) - np.log1p(dist)

    # ------------------------------------------------------------------
    # 3) Growth relative to proximity
    # ------------------------------------------------------------------

    out["growth_to_dist"] = out["radial_growth_m"] / (dist + 1)
    out["growth_rate_to_dist"] = out["radial_growth_rate_m_per_h"] / (dist + 1)

    # ------------------------------------------------------------------
    # 4) Movement / threat kinematics
    # ------------------------------------------------------------------

    # Whether there is enough perimeter history to reflect movement
    out["has_movement"] = (perimeters > 1).astype(float)

    # Indicates fire is not moving toward the zone
    out["no_closing_motion"] = (speed <= 0).astype(float)

    # Closing motion weighted by directional alignment
    out["aligned_closing"] = out["alignment_abs"] * closing_pos

    # Estimated time-to-contact using positive closing speed only
    out["eta_hours"] = np.where(closing_pos > 0.01, dist / closing_pos, 9999.0)
    out["eta_hours"] = np.clip(out["eta_hours"], 0, 9999)
    out["log_eta"] = np.log1p(out["eta_hours"])

    # Effective closing includes both motion toward zone + outward fire expansion
    effective_closing = closing_pos + radial_growth
    out["effective_closing_speed"] = effective_closing

    out["eta_effective"] = np.where(effective_closing > 0.01, dist / effective_closing, 9999.0)
    out["eta_effective"] = np.clip(out["eta_effective"], 0, 9999)
    out["log_eta_effective"] = np.log1p(out["eta_effective"])

    # Simple urgency proxy
    out["fire_urgency"] = perimeters * closing_pos

    # ------------------------------------------------------------------
    # 5) Observation quality / reliability
    # ------------------------------------------------------------------

    out["obs_density"] = perimeters / out["dt_first_last_0_5h"].clip(lower=0.1)

    out["high_quality_obs"] = (
        (perimeters >= 2) &
        (out["low_temporal_resolution_0_5h"] == 0)
    ).astype(float)

    # ------------------------------------------------------------------
    # 6) Simple temporal flags
    # ------------------------------------------------------------------

    out["is_summer"] = out["event_start_month"].isin([6, 7, 8]).astype(float)
    out["is_afternoon"] = (
        (out["event_start_hour"] >= 12) &
        (out["event_start_hour"] < 20)
    ).astype(float)

    # ------------------------------------------------------------------
    # 7) Drop redundant / noisy columns
    # ------------------------------------------------------------------
    drop_cols = [
        "relative_growth_0_5h",
        "projected_advance_m",
        "centroid_displacement_m",
        "centroid_speed_m_per_h",
        "closing_speed_abs_m_per_h",
        "area_growth_abs_0_5h",
    ]
    out = out.drop(columns=[c for c in drop_cols if c in out.columns])

    # Clean up any inf/-inf values that may have arisen from feature engineering.
    out = out.replace([np.inf, -np.inf], np.nan).fillna(0)
    return out

## III. Model

This section evaluates two complementary approaches:
- **Model A (Baseline): Penalized Cox model** for interpretable hazard-based risk ranking.
- **Model B: XGBoost horizon classifiers** for nonlinear probability estimation at 24h/48h/72h.

__Shared evaluation framework:__ Both approaches are measured with the same function `evaluate_survival_prediction`, which returns:
- Concordance index (ranking quality),
- Horizon-specific Brier scores (calibration/accuracy at 24h, 48h, 72h),
- Weighted Brier aggregate,
- Hybrid score combining ranking and probabilistic accuracy.


In [50]:
def evaluate_survival_prediction(
    y_time,
    y_event,
    risk_score,
    pred_prob_24h,
    pred_prob_48h,
    pred_prob_72h):
    """
    Evaluate survival prediction using a combination of C-index and Brier scores at multiple horizons.
    Inputs:
    - y_time: array-like of shape (n_samples,) - observed times (time_to_hit_hours)
    - y_event: array-like of shape (n_samples,) - event indicators (1 if event observed, 0 if censored)
    - risk_score: array-like of shape (n_samples,) - predicted risk scores (higher = more at risk)
    - pred_prob_24h: array-like of shape (n_samples,) - predicted probability of event by 24h
    - pred_prob_48h: array-like of shape (n_samples,) - predicted probability of event by 48h
    - pred_prob_72h: array-like of shape (n_samples,) - predicted probability of event by 72h
    Returns:
    - dict with keys "Hybrid Score", "C-Index", "Brier Score 24h", "Brier Score 48h", "Brier Score 72h", "Weighted Brier"
    """

    y_time = np.asarray(y_time)
    y_event = np.asarray(y_event).astype(int)
    risk_score = np.asarray(risk_score)

    pred_prob_24h = np.clip(np.asarray(pred_prob_24h), 0, 1)
    pred_prob_48h = np.clip(np.asarray(pred_prob_48h), 0, 1)
    pred_prob_72h = np.clip(np.asarray(pred_prob_72h), 0, 1)

    # Enforce cumulative monotonicity just in case
    pred_prob_48h = np.maximum(pred_prob_48h, pred_prob_24h)
    pred_prob_72h = np.maximum(pred_prob_72h, pred_prob_48h)

    # -------------------------
    # 1) C-index
    # lifelines expects higher predicted score = longer survival,
    # so for a risk score where larger = riskier, pass -risk_score.
    # -------------------------
    c_index = concordance_index(
        event_times=y_time,
        predicted_scores=-risk_score,
        event_observed=y_event
    )

    # -------------------------
    # 2) Horizon labels
    # Simple binary label:
    #   1 if event observed by horizon, else 0
    # -------------------------
    y_24h = ((y_event == 1) & (y_time <= 24)).astype(float)
    y_48h = ((y_event == 1) & (y_time <= 48)).astype(float)
    y_72h = ((y_event == 1) & (y_time <= 72)).astype(float)

    # -------------------------
    # 3) Brier scores
    # Mean squared error between binary outcome and predicted probability
    # -------------------------
    brier_24h = np.mean((y_24h - pred_prob_24h) ** 2)
    brier_48h = np.mean((y_48h - pred_prob_48h) ** 2)
    brier_72h = np.mean((y_72h - pred_prob_72h) ** 2)

    weighted_brier = (
        brier_24h * 0.3 +
        brier_48h * 0.4 +
        brier_72h * 0.3
    ) / (0.3 + 0.4 + 0.3)

    # -------------------------
    # 4) Hybrid score
    # Larger is better:
    #   combine C-index with (1 - weighted_brier)
    # -------------------------
    hybrid_score = 0.3 * c_index + 0.7 * (1.0 - weighted_brier)

    return {
        "Hybrid Score": hybrid_score,
        "C-Index": c_index,
        "Brier Score 24h": brier_24h,
        "Brier Score 48h": brier_48h,
        "Brier Score 72h": brier_72h,
        "Weighted Brier": weighted_brier
    }

### 1. Baseline Model: Penalized Cox model

### Overview:
The Cox model provides a strong, interpretable benchmark for survival tasks:
- Produces a principled **risk score** through the log-partial-hazard.
- Naturally handles right-censored outcomes through survival modeling.
- Helps the team understand directional feature effects before relying on more complex learners.

### Implementation details in this notebook:
- Features are engineered and split into train/validation with event stratification.
- Continuous covariates are standardized before fitting.
- `statsmodels.PHReg` is trained with **elastic-net regularization** (`L1_wt=0.0`, ridge-style penalty) to improve stability.
- Predicted survival probabilities at 24h/48h/72h are derived from baseline cumulative hazard and individual risk multipliers.

### Strengths:
- Transparent setup for debugging and communicating assumptions.
- Competitive ranking baseline for time-to-event outcomes.

### Known limitations:
- Linear predictor structure unless nonlinear terms/interactions are manually engineered.
- May underfit complex interactions that tree models capture automatically.

In [51]:
# Feature engineering for baseline
train_fe = add_time_features(train)
test_fe = add_time_features(test)

# Split dataset
target_time = "time_to_hit_hours"
target_event = "event"
id_col = "event_id"

feature_cols = [
    c for c in train_fe.columns
    if c not in [id_col, target_time, target_event]
]

X = train_fe[feature_cols].copy()
y_time = train_fe[target_time].values
y_event = train_fe[target_event].values.astype(int)

# Train test split: 80/20
X_train, X_valid, t_train, t_valid, e_train, e_valid = train_test_split(
    X, y_time, y_event,
    test_size=0.20,
    random_state=42,
    stratify=y_event
)

# Data after split
print(X_train.shape, X_valid.shape)
print("Train event rate:", e_train.mean())
print("Valid event rate:", e_valid.mean())

# Scale data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

(176, 40) (45, 40)
Train event rate: 0.3125
Valid event rate: 0.3111111111111111


In [52]:
def fit_penalized_cox(X, time, event, alpha):
    """
    Fit a penalized Cox proportional hazards model using statsmodels' PHReg with elastic net regularization.
    Parameters:
    - X: 2D array of shape (n_samples, n_features) containing the covariates.
    - time: 1D array of shape (n_samples,) containing the observed times (time to event or censoring).
    - event: 1D array of shape (n_samples,) containing the event indicators (1 if event occurred, 0 if censored).
    - alpha: float, regularization strength (higher values mean more regularization).
    Returns:
    - model: the fitted PHReg model object.
    - df: the fitted model dfs object containing parameter estimates and statistics.
    """
    
    model = PHReg(
        endog=time,
        exog=X,
        status=event,
        ties="breslow"
    )

    df = model.fit_regularized(
        method="elastic_net",
        alpha=alpha,
        L1_wt=0.0,   
        maxiter=1000,
        cnvrg_tol=1e-8,
        zero_tol=1e-10
    )
    return model, df

cox_model, cox_df = fit_penalized_cox(
    X_train_scaled,
    t_train,
    e_train,
    alpha=0.1
)

In [53]:
def get_baseline_cumulative_hazard(model, df):
    bch = model.baseline_cumulative_hazard(df.params)
    times = np.asarray(bch[0][0])
    cumhaz = np.asarray(bch[0][1])
    return times, cumhaz

def cumulative_hazard_at_times(base_times, base_cumhaz, query_times):
    query_times = np.asarray(query_times)
    idx = np.searchsorted(base_times, query_times, side="right") - 1
    idx = np.clip(idx, 0, len(base_cumhaz) - 1)
    values = base_cumhaz[idx]
    values = np.where(query_times < base_times[0], 0.0, values)
    return values

def predict_event_probabilities(model, df, X_scaled, horizons=(24, 48, 72)):
    base_times, base_cumhaz = get_baseline_cumulative_hazard(model, df)

    lp = np.asarray(X_scaled @ df.params).reshape(-1)
    risk_multiplier = np.exp(lp)

    out = {}
    for h in horizons:
        H0_h = cumulative_hazard_at_times(base_times, base_cumhaz, [h])[0]
        survival_h = np.exp(-H0_h * risk_multiplier)
        out[f"prob_{h}h"] = 1 - survival_h

    return out

In [54]:
valid_probs = predict_event_probabilities(
    model=cox_model,
    df=cox_df,
    X_scaled=X_valid_scaled,
    horizons=(24, 48, 72)
)

valid_risk_score = np.asarray(X_valid_scaled @ cox_df.params).reshape(-1)

metrics = evaluate_survival_prediction(
    y_time=t_valid,
    y_event=e_valid,
    risk_score=valid_risk_score,
    pred_prob_24h=valid_probs["prob_24h"],
    pred_prob_48h=valid_probs["prob_48h"],
    pred_prob_72h=valid_probs["prob_72h"]
)

metrics

{'Hybrid Score': np.float64(0.8802789627828649),
 'C-Index': np.float64(0.8692946058091287),
 'Brier Score 24h': np.float64(0.11414672641235557),
 'Brier Score 48h': np.float64(0.11701987019574073),
 'Brier Score 72h': np.float64(0.11320496551653157),
 'Weighted Brier': np.float64(0.11501345565696243)}

## 2. XGBoost

### Overview
XGBoost is used to capture nonlinear relationships and higher-order interactions that the Cox baseline may miss.

### Framing used here
Instead of fitting a single survival learner, the notebook trains **three binary classifiers**:
- Event by 24 hours,
- Event by 48 hours,
- Event by 72 hours.

### Pipeline summary
- Build horizon-specific binary labels from `(time_to_hit_hours, event)`.
- Use a shared train/validation split (stratified by event occurrence).
- Train one `XGBClassifier` per horizon with class-imbalance reweighting (`scale_pos_weight`).
- Convert outputs to probabilities and enforce temporal monotonicity:
  - `P(event by 48h) >= P(event by 24h)`
  - `P(event by 72h) >= P(event by 48h)`
- Create a composite risk score as a weighted combination of horizon probabilities for C-index evaluation.

### Why monotonicity is enforced
Cumulative event probability should not decrease as the prediction horizon gets longer. Enforcing this rule ensures outputs remain temporally coherent and easier to interpret.

### Evaluation
Final XGBoost validation metrics are computed with the same `evaluate_survival_prediction` function used for the Cox baseline, enabling direct side-by-side comparison.



In [55]:
# Apply feature engineering
train_new = create_features(train)
test_new = create_features(test)

# Prepare data for XGBoost
feature_cols = [
    c for c in train_new.columns
    if c not in [id_col, target_time, target_event]
]

X = train_new[feature_cols].copy()
y_time = train_new[target_time].values
y_event = train_new[target_event].values.astype(int)

In [61]:
# Create horizon labels for XGBoost
def make_horizon_label(time, event, horizon):
    """
    Binary label:
    1 if event observed by horizon, else 0
    """
    return ((event == 1) & (time <= horizon)).astype(int)

# Apply horizon labeling
y_12 = make_horizon_label(y_time, y_event, 12)
y_24 = make_horizon_label(y_time, y_event, 24)
y_48 = make_horizon_label(y_time, y_event, 48)
y_72 = make_horizon_label(y_time, y_event, 72)

In [68]:
# Train test split: 80/20
idx = np.arange(len(X))

idx_train, idx_valid = train_test_split(
    idx,
    test_size=0.20,
    random_state=42,
    stratify=y_event
)

X_train = X.iloc[idx_train].copy()
X_valid = X.iloc[idx_valid].copy()

t_train = y_time[idx_train]
t_valid = y_time[idx_valid]

e_train = y_event[idx_train]
e_valid = y_event[idx_valid]

y12_train, y12_valid = y_12[idx_train], y_12[idx_valid]
y24_train, y24_valid = y_24[idx_train], y_24[idx_valid]
y48_train, y48_valid = y_48[idx_train], y_48[idx_valid]
y72_train, y72_valid = y_72[idx_train], y_72[idx_valid]

print("Train shape:", X_train.shape)
print("Valid shape:", X_valid.shape)
print("Train event rate:", e_train.mean())
print("Valid event rate:", e_valid.mean())

Train shape: (176, 56)
Valid shape: (45, 56)
Train event rate: 0.3125
Valid event rate: 0.3111111111111111


In [69]:
def train_xgb_horizon_model(X_train, y_train, X_valid, y_valid, random_state=42):
    """
    Train one binary XGBoost classifier for a single horizon.
    """
    pos = max(y_train.sum(), 1)
    neg = max((y_train == 0).sum(), 1)
    scale_pos_weight = neg / pos

    model = XGBClassifier(
        objective="binary:logistic",
        n_estimators=800,
        max_depth=8,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_alpha=0.0,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        random_state=random_state,
        eval_metric="logloss",
        tree_method="hist"
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False
    )

    return model

In [71]:
# Train one model per horizon
model_12 = train_xgb_horizon_model(X_train, y12_train, X_valid, y12_valid, random_state=41)
model_24 = train_xgb_horizon_model(X_train, y24_train, X_valid, y24_valid, random_state=42)
model_48 = train_xgb_horizon_model(X_train, y48_train, X_valid, y48_valid, random_state=43)
model_72 = train_xgb_horizon_model(X_train, y72_train, X_valid, y72_valid, random_state=44)

# Predict probabilities on validation
pred_12_valid = model_12.predict_proba(X_valid)[:, 1]
pred_24_valid = model_24.predict_proba(X_valid)[:, 1]
pred_48_valid = model_48.predict_proba(X_valid)[:, 1]
pred_72_valid = model_72.predict_proba(X_valid)[:, 1]

# Enforce cumulative monotonicity
pred_24 = np.maximum(pred_24_valid, pred_12_valid)
pred_48 = np.maximum(pred_48_valid, pred_24)
pred_72 = np.maximum(pred_72_valid, pred_48)

# Final clipping just to be safe
pred_12 = np.clip(pred_12_valid, 0, 1)
pred_24 = np.clip(pred_24, 0, 1)
pred_48 = np.clip(pred_48, 0, 1)
pred_72 = np.clip(pred_72, 0, 1)

In [72]:
# Computer risk score for C-index evaluation.
# Use a weighted combination of horizon probabilities
# Heavier weight on nearer horizons -> more "urgent" fires get higher risk scores
risk_score_valid = (
    0.5 * pred_24 +
    0.3 * pred_48 +
    0.2 * pred_72
)

In [73]:
metrics = evaluate_survival_prediction(
    y_time=t_valid,
    y_event=e_valid,
    risk_score=risk_score_valid,
    pred_prob_24h=pred_24,
    pred_prob_48h=pred_48,
    pred_prob_72h=pred_72
)

metrics

{'Hybrid Score': np.float64(0.966646153085952),
 'C-Index': np.float64(0.921161825726141),
 'Brier Score 24h': np.float64(0.043583473744080295),
 'Brier Score 48h': np.float64(0.0013007500173664195),
 'Brier Score 72h': np.float64(0.0008840720989086152),
 'Weighted Brier': np.float64(0.01386056375984324)}

## 3. Save Submissions to Submit on Kaggle

In [77]:
# Prepare Kaggle test features
X_test = test_new[feature_cols].copy()

# Predict probabilities on test set
pred_12_test = model_12.predict_proba(X_test)[:, 1]
pred_24_test = model_24.predict_proba(X_test)[:, 1]
pred_48_test = model_48.predict_proba(X_test)[:, 1]
pred_72_test = model_72.predict_proba(X_test)[:, 1]

# Enforce cumulative monotonicity
pred_24_test = np.maximum(pred_24_test, pred_12_test)
pred_48_test = np.maximum(pred_48_test, pred_24_test)
pred_72_test = np.maximum(pred_72_test, pred_48_test)

# Clip to [0, 1]
pred_12_test = np.clip(pred_12_test, 0, 1)
pred_24_test = np.clip(pred_24_test, 0, 1)
pred_48_test = np.clip(pred_48_test, 0, 1)
pred_72_test = np.clip(pred_72_test, 0, 1)

In [78]:
submission = pd.DataFrame({
    "event_id": test["event_id"].values,
    "prob_12h": pred_12_test,
    "prob_24h": pred_24_test,
    "prob_48h": pred_48_test,
    "prob_72h": pred_72_test,
})

submission.head()

,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.023616,0.023616,0.023616,0.028554
1,13353600,0.788154,0.982887,0.983221,0.983221
2,13942327,0.011962,0.017663,0.017663,0.028554
3,16112781,0.919919,0.982080,0.982080,0.982080
4,17132808,0.011668,0.015073,0.015073,0.028554


In [79]:
submission.to_csv("../data/submission.csv", index=False)
print("Saved submission.csv")

Saved submission.csv
